In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# 1. SELECCIONAR DADES (Selección de datos)
# Cargamos el dataset y separamos las características del objetivo según la estructura detectada
df = pd.read_csv("../data/data_original/house_pricing/house_pricing.csv")

In [3]:
# Separamos el conjunto de entrenamiento (labeled) y el de test del leaderboard
df_labeled = df[df['Split'] == 'labeled'].copy()
df_leaderboard = df[df['Split'] == 'leaderboard'].copy()

# Definimos la variable objetivo y eliminamos columnas que no aportan valor predictivo directo (Id, Split)
X = df_labeled.drop(columns=['Split', 'Id', 'SalePrice'])
y = df_labeled['SalePrice']

X_leaderboard = df_leaderboard.drop(columns=['Split', 'Id', 'SalePrice'])

# Dividimos en entrenamiento y validación interna para asegurar la robustez del modelo
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# 2. NETEJAR DADES (Limpieza de datos) y 5. FORMATAR DADES (Feature Engineering)
# La guía indica que no se deben eliminar/imputar nulos sistemáticamente a menos que sea necesario o se conozca el valor real.
# En este dataset, los nulos en categorías de Garaje o Sótano suelen significar la 'Ausencia' de dicha característica.

# Identificación de columnas numéricas y categóricas
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Corrección de errores tipográficos explícitos detectados en las columnas (Ej: 'Electtrical' vs 'Electrical')
if 'Electtrical' in categorical_cols:
    # Si contiene datos duplicados o erróneos se unifica o elimina según la guía
    X_train = X_train.drop(columns=['Electtrical'])
    X_val = X_val.drop(columns=['Electtrical'])
    X_leaderboard = X_leaderboard.drop(columns=['Electtrical'])
    if 'Electtrical' in categorical_cols: categorical_cols.remove('Electtrical')

# Pipeline para variables numéricas:
# Corregimos nulos con mediana solo si es necesario, y aplicamos escalado estándar
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())
])

# Pipeline para variables categóricas:
# Tratamos nulos asignando una categoría constante ('Missing'/'None') que preserva el valor real de su ausencia
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinamos los transformadores en un preprocesador unificado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

In [ ]:
# 3. CONSTRUIR DADES (Ingeniería de Características avanzada)
# La guía sugiere generar valor construyendo nuevas columnas relevantes antes del formateo final
def construir_caracteristicas(df_in):
    df_out = df_in.copy()
    
    # Combinación de áreas para obtener los metros cuadrados totales de la vivienda
    if '1stFlrSF' in df_out.columns and '2ndFlrSF' in df_out.columns:
        df_out['TotalSuperficieViva'] = df_out['1stFlrSF'] + df_out['2ndFlrSF'] + df_out.get('GrLivArea', 0)
        
    # Edad de la propiedad en el momento de la última transacción o remodelación
    if 'YearBuilt' in df_out.columns and 'YearRemodAdd' in df_out.columns:
        df_out['AnosDesdeConstruccion'] = df_out['YearRemodAdd'] - df_out['YearBuilt']
        
    # Total de baños completos disponibles en la casa
    if 'FullBath' in df_out.columns and 'BsmtFullBath' in df_out.columns:
        df_out['TotalBanosCompletos'] = df_out['FullBath'] + df_out['BsmtFullBath']
        
    return df_out

# Aplicamos la fase de construcción de datos en nuestros conjuntos
X_train_built = construir_caracteristicas(X_train)
X_val_built = construir_caracteristicas(X_val)
X_leaderboard_built = construir_caracteristicas(X_leaderboard)

# Actualizamos las listas de columnas tras la construcción
numerical_cols_updated = X_train_built.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_updated = X_train_built.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor.set_params(
    num__numerical_cols=numerical_cols_updated,
    cat__categorical_cols=categorical_cols_updated
)

# Ajustamos y transformamos de manera limpia para evitar Data Leakage
X_train_ready = preprocessor.fit_transform(X_train_built)
X_val_ready = preprocessor.transform(X_val_built)
X_leaderboard_ready = preprocessor.transform(X_leaderboard_built)

print(f"Forma del set de entrenamiento final: {X_train_ready.shape}")